# BERT-base-uncased — Codenames Duet layer extraction

This notebook is a thin orchestration shell over the `codenames` package. It runs the same experiment defined in the package's modules; methodology lives in the package, not here.

After the shared Colab setup (Cells 1–3: clone, install, mount), run the cells below in order: load model → load dataset → run extraction → SC1 → ... → SC7 → output summary, each in its own cell. The extraction checkpoints to Drive every ~200 boards and is resumable (`RESUME`), so if the Colab runtime dies you can re-run the extraction cell to continue from where it stopped.

## Setup (run cells 1–3 once per session)

In [ ]:
# Cell 1 — Clone or update the package code from GitHub.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Cell 2 — Install the package in editable mode.
!pip install -q -e {REPO_DIR}

In [ ]:
# Cell 2b — Verify the installed environment matches the pinned, reproducible
# dependency set (read-only; downloads no weights). Stop and fix if this fails.
!codenames-experiment doctor --model bert

In [ ]:
# Cell 3 — Autoreload (so package edits flow through without restarting) and mount Drive.
import sys, types, importlib
if "imp" not in sys.modules:
    # Python 3.12 removed `imp`; old IPython autoreload still imports it.
    _imp = types.ModuleType("imp")
    _imp.reload = importlib.reload
    sys.modules["imp"] = _imp

# Put the freshly cloned package on sys.path of THIS kernel. `pip install -e .`
# exposes it via a .pth file Python only reads at interpreter startup, so a
# kernel started before Cell 2 can't `import codenames` until we add the repo
# root here. (Path A works without this because it runs in a fresh subprocess.)
REPO_DIR = "/content/codenames-interpretability"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# === Run size ===
# SAMPLE_SIZE controls how many boards to process:
#   None           -> the FULL dataset (all 7704 boards in clue_generation.csv)
#   <int>, e.g. 50 -> a reproducible random sample of that many boards (quick test)
# The frozen contract baseline is N=2000 (CONTRACT_V1.sample_size); the load/sample
# cell below overrides it with this run size, so the full dataset is processed.
SAMPLE_SIZE = None
print("Run size:", "FULL dataset (7704 boards)" if SAMPLE_SIZE is None else f"{SAMPLE_SIZE} boards")

## Extraction and sanity checks

Run the cells below one at a time. Each produces its own output for inspection.

In [ ]:
# Cell 4 — Imports and config.
import dataclasses
from codenames.contract import CONTRACT_V1, Acceleration
from codenames.data import (
    load_dataset, sample_turns, GIVER_COLS, extract_giver_features,
)
from codenames.models.bert import load_bert_base
from codenames.loop import run_extraction
from codenames.sanity import (
    sc1_prompt_structure, sc2_span_coverage, sc3_anisotropy,
    sc4_behavioral_accuracy, sc5_layer_margin_curve,
    sc6_positional_confound, sc7_shuffle_decomposition,
)
from codenames.persistence import print_output_summary

DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"

# --- Acceleration settings ---
# BERT-base (encoder-only): vectorized anisotropy + board batching. No FA2
# (encoder-only model class doesn't use the causal FA2 path). 110M params
# allow batch=32 on an A100/L4 (drop to 16 on a T4).
ACCEL = Acceleration(
    vectorize_anisotropy=True,
    batch_size=32,
)

In [ ]:
# Cell 5 — Load model.
model, tokenizer, meta = load_bert_base()
print(f"Model loaded: {meta['model_name']}")
print(f"  Layers: {meta['num_layers']}, Hidden dim: {meta['hidden_dim']}")

In [ ]:
# Cell 6 — Load and sample dataset.
df = load_dataset(DATASET_PATH)
n_boards = len(df) if SAMPLE_SIZE is None else SAMPLE_SIZE
# Build the run contract from the frozen baseline with sample_size set to
# the requested run size, so run_extraction processes n_boards (NOT the
# N=2000 contract default). Mirrors the CLI's --full / --sample-size.
CONTRACT = dataclasses.replace(CONTRACT_V1, sample_size=n_boards)
df_sample = sample_turns(df, n=CONTRACT.sample_size, seed=CONTRACT.random_seed)
print(f"Sample size: {len(df_sample)} boards")
print(f"First 10 row_ids: {sorted(df_sample['row_id'].tolist())[:10]}")

In [ ]:
# Cell 7 — Run extraction.
BASE_DIR = f"/content/drive/MyDrive/TCC/{meta['prefix']}_outputs"
# Intermediate checkpoint/manifest files (and the reuse cache, if enabled)
# go in a SEPARATE folder, so BASE_DIR holds only the final result files.
CHECKPOINT_DIR = f"/content/drive/MyDrive/TCC/{meta['prefix']}_checkpoints"

# Checkpoint / resume — ENABLED. Outputs are checkpointed to BASE_DIR every
# ~200 boards. The first run starts fresh; if the Colab runtime dies mid-run,
# just re-run this cell and it continues from the last checkpoint (skipping
# already-committed boards). A resumed run is byte-identical to an
# uninterrupted one. Set RESUME = False to force a clean restart that wipes
# any existing checkpoints first.
RESUME = True

# Cross-size canonical reuse — OFF. This notebook batches for speed (ACCEL
# below has batch_size>1), and reuse only works on the reference path
# (batch_size=1), so it stays disabled here. To use it instead, set
# ACCEL = Acceleration() (batch_size=1) AND REUSE_CANONICAL = True on both
# the smaller and the larger run; cached per-board canonical
# (permutation_id=0) results are then reused across SAMPLE_SIZE (shuffles
# always recomputed), byte-identical to a non-reusing run.
REUSE_CANONICAL = False

results = run_extraction(
    model=model,
    tokenizer=tokenizer,
    df=df_sample,
    base_dir=BASE_DIR,
    prefix=meta["prefix"],
    contract=CONTRACT,
    chat_template_strategy=meta["chat_template_strategy"],
    forward_hidden_states_mode=meta["forward_hidden_states_mode"],
    use_truncation=meta["use_truncation"],
    num_layers=meta["num_layers"],
    hidden_dim=meta["hidden_dim"],
    device=meta["device"],
    has_generation=meta["supports_generation"],
    generation_fn=None,  # encoder-only
    acceleration=ACCEL,
    resume=RESUME,
    reuse_canonical=REUSE_CANONICAL,
    checkpoint_dir=CHECKPOINT_DIR,
)

In [ ]:
# Cell 8 — SC1.
sc1_prompt_structure(df_sample, tokenizer, meta["chat_template_strategy"])

In [ ]:
# Cell 9 — SC2.
sc2_span_coverage(results)

In [ ]:
# Cell 10 — SC3.
sc3_anisotropy(results, num_layers=meta["num_layers"])

In [ ]:
# Cell 11 — SC4.
sc4_behavioral_accuracy(
    results,
    pooling_methods=CONTRACT_V1.pooling_methods,
    has_generation=meta["supports_generation"],
)

In [ ]:
# Cell 12 — SC5.
sc5_layer_margin_curve(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"], pooling_methods=CONTRACT_V1.pooling_methods,
)

In [ ]:
# Cell 13 — SC6.
sc6_positional_confound(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"],
)

In [ ]:
# Cell 14 — SC7.
sc7_shuffle_decomposition(
    results, base_dir=BASE_DIR, prefix=meta["prefix"],
    num_layers=meta["num_layers"], n_shuffles=CONTRACT_V1.n_shuffles,
)

In [ ]:
# Cell 15 — Output summary.
print_output_summary(
    base_dir=BASE_DIR, prefix=meta["prefix"], contract=CONTRACT_V1,
    has_generation=meta["supports_generation"],
    pooling_methods=CONTRACT_V1.pooling_methods,
)